# Intermediate Topics in LangGraph

## Tutorials Covered:
1. Multi-Agent Systems
2. Human-in-the-Loop
3. Streaming Responses
4. State Persistence

## 1. Multi-Agent Systems

### Learning Objectives:
- Design and implement multi-agent systems
- Coordinate multiple agents in a single graph
- Handle inter-agent communication

Multi-agent systems allow multiple specialized agents to work together to solve complex problems.

In [ ]:
# Example of multi-agent system
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph

# Define state for multi-agent system
class MultiAgentState(TypedDict):
    task: str
    research_result: str
    writing_result: str
    review_result: str
    final_output: str
    agent_messages: Annotated[list, operator.add]
    current_agent: str

# Research agent
def research_agent(state):
    task = state['task']
    research_result = f"Research completed on: {task}. Key findings: LangGraph enables stateful multi-step applications."
    
    return {
        "research_result": research_result,
        "agent_messages": [f"Research agent: {research_result}"],
        "current_agent": "research"
    }

# Writing agent
def writing_agent(state):
    research = state['research_result']
    writing_result = f"Article draft based on research: {research[:50]}... [draft content]"
    
    return {
        "writing_result": writing_result,
        "agent_messages": [f"Writing agent: {writing_result}"],
        "current_agent": "writer"
    }

# Review agent
def review_agent(state):
    draft = state['writing_result']
    review_result = f"Review completed: {draft[:30]}... [feedback and suggestions]"
    
    return {
        "review_result": review_result,
        "agent_messages": [f"Review agent: {review_result}"],
        "current_agent": "reviewer"
    }

# Synthesis agent that combines results
def synthesis_agent(state):
    research = state['research_result']
    writing = state['writing_result']
    review = state['review_result']
    
    final_output = f"FINAL REPORT:\nResearch: {research}\nDraft: {writing}\nReview: {review}"
    
    return {
        "final_output": final_output,
        "agent_messages": ["Synthesis agent: Final report compiled"],
        "current_agent": "synthesis"
    }

# Create multi-agent workflow
multi_agent_workflow = StateGraph(MultiAgentState)

multi_agent_workflow.add_node("researcher", research_agent)
multi_agent_workflow.add_node("writer", writing_agent)
multi_agent_workflow.add_node("reviewer", review_agent)
multi_agent_workflow.add_node("synthesizer", synthesis_agent)

multi_agent_workflow.set_entry_point("researcher")
multi_agent_workflow.add_edge("researcher", "writer")
multi_agent_workflow.add_edge("writer", "reviewer")
multi_agent_workflow.add_edge("reviewer", "synthesizer")
multi_agent_workflow.add_edge("synthesizer", "__end__")

multi_agent_app = multi_agent_workflow.compile()

# Test the multi-agent system
multi_agent_result = multi_agent_app.invoke({
    "task": "Explain the benefits of LangGraph",
    "research_result": "",
    "writing_result": "",
    "review_result": "",
    "final_output": "",
    "agent_messages": [],
    "current_agent": ""
})

print("Multi-agent system result:")
print(multi_agent_result['final_output'])

## 2. Human-in-the-Loop

### Learning Objectives:
- Implement human feedback points in workflows
- Design interfaces for human interaction
- Handle human decisions in automated processes

Human-in-the-loop systems allow humans to intervene and make decisions in automated workflows.

In [ ]:
# Example of human-in-the-loop system
import time

# State for human-in-the-loop system
class HumanInLoopState(TypedDict):
    input_request: str
    automated_analysis: str
    human_feedback: str
    final_decision: str
    requires_human_input: bool
    human_response_received: bool

# Automated analysis node
def automated_analysis_node(state):
    input_req = state['input_request']
    analysis = f"Automated analysis of '{input_req}': This appears to be a standard request that can be processed automatically."
    
    # Determine if human input is needed (simple rule for demo)
    needs_human = "urgent" in input_req.lower() or "critical" in input_req.lower()
    
    return {
        "automated_analysis": analysis,
        "requires_human_input": needs_human
    }

# Human input node (simulated)
def human_input_node(state):
    analysis = state['automated_analysis']
    print(f"\n*** HUMAN INPUT REQUIRED ***\n")
    print(f"Automated analysis: {analysis}")
    print("Please provide feedback or approval: ")
    
    # Simulate human input
    human_feedback = "Request approved with minor changes suggested"
    print(f"Simulated human feedback: {human_feedback}")
    
    return {
        "human_feedback": human_feedback,
        "human_response_received": True
    }

# Final decision node
def final_decision_node(state):
    if state.get('human_response_received', False):
        decision = f"Final decision based on human feedback: {state['human_feedback']}"
    else:
        decision = f"Proceeding automatically based on analysis: {state['automated_analysis']}"
    
    return {"final_decision": decision}

# Create human-in-the-loop workflow
hil_workflow = StateGraph(HumanInLoopState)

hil_workflow.add_node("analysis", automated_analysis_node)
hil_workflow.add_node("human_input", human_input_node)
hil_workflow.add_node("decision", final_decision_node)

hil_workflow.set_entry_point("analysis")

# Add conditional routing based on whether human input is needed
def route_based_on_human_needed(state):
    if state.get('requires_human_input', False):
        return "human_input"
    else:
        return "decision"

hil_workflow.add_conditional_edges(
    "analysis",
    route_based_on_human_needed,
    {
        "human_input": "human_input",
        "decision": "decision"
    }
)

hil_workflow.add_edge("human_input", "decision")
hil_workflow.add_edge("decision", "__end__")

hil_app = hil_workflow.compile()

# Test with regular request (no human input needed)
regular_result = hil_app.invoke({
    "input_request": "Can you explain how LangGraph works?",
    "automated_analysis": "",
    "human_feedback": "",
    "final_decision": "",
    "requires_human_input": False,
    "human_response_received": False
})
print("Regular request result:", regular_result['final_decision'])

# Test with urgent request (human input needed)
urgent_result = hil_app.invoke({
    "input_request": "This is an urgent critical issue with our production system!",
    "automated_analysis": "",
    "human_feedback": "",
    "final_decision": "",
    "requires_human_input": False,
    "human_response_received": False
})
print("\nUrgent request result:", urgent_result['final_decision'])

## 3. Streaming Responses

### Learning Objectives:
- Implement streaming responses in LangGraph
- Handle incremental output delivery
- Manage partial results in workflows

Streaming responses provide real-time feedback to users during long-running operations.

In [ ]:
# Example of streaming responses in LangGraph
import asyncio
from typing import Generator

# State for streaming system
class StreamingState(TypedDict):
    query: str
    streaming_output: Annotated[list, operator.add]
    final_result: str
    stream_complete: bool

# Simulated streaming node
def streaming_processor_node(state):
    query = state['query']
    
    # Simulate streaming response
    chunks = [
        f"Processing your request: '{query}'",
        "Step 1: Analyzing query...",
        "Step 2: Gathering information...",
        "Step 3: Formulating response...",
        "Step 4: Finalizing output..."
    ]
    
    stream_parts = []
    for i, chunk in enumerate(chunks):
        # Simulate delay for streaming effect
        time.sleep(0.5)
        stream_parts.append(chunk)
        print(f"Streamed: {chunk}")
    
    final_result = f"Complete response for query: '{query}'. Processed {len(chunks)} steps."
    
    return {
        "streaming_output": stream_parts,
        "final_result": final_result,
        "stream_complete": True
    }

# Alternative approach: Using a generator-like pattern
def streaming_generator_node(state):
    query = state['query']
    
    # Simulate generating streaming output
    parts = [
        f"Generating insights for: {query}",
        "Accessing knowledge base...",
        "Performing analysis...",
        "Compiling results..."
    ]
    
    # Return each part as it's generated
    for part in parts:
        print(f"Yielding: {part}")
        yield {"streaming_output": [part]}
    
    # Finally return the complete result
    final = f"Final answer for '{query}': LangGraph is a powerful framework for building stateful, multi-step AI applications."
    return {
        "final_result": final,
        "stream_complete": True
    }

# Create streaming workflow
streaming_workflow = StateGraph(StreamingState)

streaming_workflow.add_node("processor", streaming_processor_node)

streaming_workflow.set_entry_point("processor")
streaming_workflow.add_edge("processor", "__end__")

streaming_app = streaming_workflow.compile()

# Test streaming response
print("Testing streaming response:")
streaming_result = streaming_app.invoke({
    "query": "How does LangGraph differ from regular LLM chains?",
    "streaming_output": [],
    "final_result": "",
    "stream_complete": False
})

print(f"\nFinal streamed result: {streaming_result['final_result']}")
print(f"Total stream parts received: {len(streaming_result['streaming_output'])}")

## 4. State Persistence

### Learning Objectives:
- Implement state persistence mechanisms
- Save and restore graph state
- Handle long-running workflows

State persistence allows workflows to save their progress and resume later, which is essential for long-running processes.

In [ ]:
# Example of state persistence in LangGraph
import json
import os
from uuid import uuid4

# State for persistence system
class PersistentState(TypedDict):
    task_id: str
    current_step: str
    step_results: Annotated[list, operator.add]
    completed_steps: Annotated[list, operator.add]
    workflow_status: str
    resume_token: str

# Simulated long-running workflow steps
def step_one_node(state):
    task_id = state['task_id']
    print(f"Executing step 1 for task {task_id}")
    
    # Simulate some work
    result = f"Step 1 completed for task {task_id}"
    
    return {
        "current_step": "step_one",
        "step_results": [result],
        "completed_steps": ["step_one"],
        "workflow_status": "in_progress"
    }

def step_two_node(state):
    task_id = state['task_id']
    print(f"Executing step 2 for task {task_id}")
    
    # Simulate some work
    result = f"Step 2 completed for task {task_id}"
    
    return {
        "current_step": "step_two",
        "step_results": [result],
        "completed_steps": ["step_two"],
        "workflow_status": "in_progress"
    }

def step_three_node(state):
    task_id = state['task_id']
    print(f"Executing step 3 for task {task_id}")
    
    # Simulate some work
    result = f"Step 3 completed for task {task_id}"
    
    return {
        "current_step": "step_three",
        "step_results": [result],
        "completed_steps": ["step_three"],
        "workflow_status": "completed"
    }

# Persistence helper functions
def save_state_to_file(state, filename):
    """Save state to a file"""
    with open(filename, 'w') as f:
        json.dump(state, f, indent=2)
    print(f"State saved to {filename}")

def load_state_from_file(filename):
    """Load state from a file"""
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            state = json.load(f)
        print(f"State loaded from {filename}")
        return state
    else:
        print(f"State file {filename} not found")
        return None

# Create persistent workflow
persistent_workflow = StateGraph(PersistentState)

persistent_workflow.add_node("step_one", step_one_node)
persistent_workflow.add_node("step_two", step_two_node)
persistent_workflow.add_node("step_three", step_three_node)

persistent_workflow.set_entry_point("step_one")
persistent_workflow.add_edge("step_one", "step_two")
persistent_workflow.add_edge("step_two", "step_three")
persistent_workflow.add_edge("step_three", "__end__")

persistent_app = persistent_workflow.compile()

# Simulate a workflow that gets interrupted and resumed
task_id = str(uuid4())
state_file = f"/workspace/workflow_state_{task_id}.json"

# Initial state
initial_state = {
    "task_id": task_id,
    "current_step": "",
    "step_results": [],
    "completed_steps": [],
    "workflow_status": "started",
    "resume_token": ""
}

# Execute first step
print("Executing step one...")
result_step1 = persistent_app.invoke(initial_state)
print(f"After step 1: {result_step1['step_results'][-1]}")

# Simulate interruption and save state
save_state_to_file(result_step1, state_file)
print("Workflow interrupted, state saved.")

# Later, resume workflow from saved state
print("\nResuming workflow from saved state...")
saved_state = load_state_from_file(state_file)

# For this example, we'll just continue with the rest of the workflow
# In a real scenario, you might need to determine which step to continue from
if saved_state:
    # Continue with remaining steps
    print("Continuing with step two and three...")
    result_remaining = persistent_app.invoke(saved_state)
    
    print(f"\nWorkflow completed. Final status: {result_remaining['workflow_status']}")
    print(f"All completed steps: {result_remaining['completed_steps']}")
    print(f"All results: {result_remaining['step_results']}")

# Clean up
if os.path.exists(state_file):
    os.remove(state_file)
    print(f"\nCleaned up state file: {state_file}")

## Practice Exercises

1. Create a multi-agent system for software development with coding, testing, and documentation agents
2. Implement a human-in-the-loop approval workflow for sensitive operations
3. Design a streaming response system that provides real-time progress updates
4. Build a persistent workflow for processing large datasets that takes hours to complete

## Additional Resources

- [Multi-Agent Systems in LangGraph](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/)
- [Human-in-the-Loop Patterns](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/)
- [Streaming Responses Guide](https://langchain-ai.github.io/langgraph/how-tos/streaming/)
- [State Persistence Best Practices](https://langchain-ai.github.io/langgraph/how-tos/persistence/)